<a href="https://colab.research.google.com/github/emilheroyt/Bachelor-Thesis-Stress-Testing-Robustness-of-SOTA-Point-Trackers/blob/main/experiment2/cowtracker_Experiment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Zelle 1 – GPU-Check:



In [1]:
!nvidia-smi

Mon Sep 21 20:51:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   44C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Zelle 2 – Repo (ZIP-Weg), Submodule, Pakete, FlashAttention-Patch

In [2]:
%cd /content
!rm -rf cowtracker main.zip
!wget -q https://github.com/facebookresearch/cowtracker/archive/refs/heads/main.zip
!unzip -q main.zip && mv cowtracker-main cowtracker
%cd /content/cowtracker/cowtracker/thirdparty
!rm -rf DepthAnythingV2 vggt
!wget -q https://github.com/DepthAnything/Depth-Anything-V2/archive/refs/heads/main.zip -O da2.zip && unzip -q da2.zip && mv Depth-Anything-V2-main DepthAnythingV2
!wget -q https://github.com/facebookresearch/vggt/archive/refs/heads/main.zip -O vggt.zip && unzip -q vggt.zip && mv vggt-main vggt
%cd /content/cowtracker
!pip install mediapy xformers -q

import re
path = "/content/cowtracker/cowtracker/layers/video_transformer.py"
with open(path) as f:
    content = f.read()
pattern = r"(def forward\(\s*self, x: torch\.Tensor, attn_mask: torch\.Tensor \| None = None)(\s*\) -> torch\.Tensor:)"
new_content, n = re.subn(pattern, r"\1, is_causal: bool = False, **kwargs\2", content)
with open(path, "w") as f:
    f.write(new_content)
print(f"FlashAttention-Patch: {n} Stelle(n) ersetzt")

/content
/content/cowtracker/cowtracker/thirdparty
/content/cowtracker
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 132.6 MB/s eta 0:00:00
FlashAttention-Patch: 1 Stelle(n) ersetzt


Zelle 3 – TAP-Vid-DAVIS + Eval-Utility (mit TensorFlow-Umgehung)

In [3]:
!wget -q https://storage.googleapis.com/dm-tapnet/tapvid_davis.zip && unzip -q -o tapvid_davis.zip
!wget -q https://raw.githubusercontent.com/google-deepmind/tapnet/refs/heads/main/tapnet/tapvid/evaluation_datasets.py
!wget -q https://github.com/google-deepmind/tapnet/archive/refs/heads/main.zip -O tapnet_repo.zip && unzip -q -o tapnet_repo.zip
!pip install "./tapnet-main[torch]" -q

import sys
from unittest.mock import MagicMock
sys.modules['tensorflow'] = MagicMock()
sys.modules['tensorflow_datasets'] = MagicMock()
sys.path.insert(0, '.')
import evaluation_datasets
from evaluation_datasets import create_davis_dataset, compute_tapvid_metrics
evaluation_datasets.tf.io.gfile.GFile = open
import numpy as np
print("TAP-Vid bereit:", __import__("os").path.exists("tapvid_davis/tapvid_davis.pkl"))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
TAP-Vid bereit: True


Zelle 4 – Modell laden (TensorFlow-Attrappe vorher entfernen)

In [4]:
import sys
for m in ["tensorflow", "tensorflow_datasets"]:
    sys.modules.pop(m, None)
for m in [k for k in sys.modules if k == "cowtracker" or k.startswith("cowtracker.")]:
    del sys.modules[m]
sys.path.insert(0, "/content/cowtracker")

import torch
import torch.nn.functional as F
from cowtracker import CoWTracker
model = CoWTracker.from_checkpoint(device="cuda", dtype=torch.float16)

timm version:  1.0.29
Initializing CoWTracker...
✓ Flash Attention 3 enabled for spatial attention: replaced 12 attention modules
CowTrackingHead initialized: iter_dim=64, warp_iters=5
  - Features: 128, Side channels: 128
  - Warping-based iterative refinement iterations: 5


cowtracker_model.pth: reconstructing file:   0%|          |  0.00B / 3.90GB            

cowtracker_model.pth: downloading bytes:           |  0.00B            

Downloaded to: /root/.cache/huggingface/hub/models--facebook--cowtracker/snapshots/860b85cfb3a63b91780535689c262b3e595b6082/cowtracker_model.pth
Detected legacy checkpoint format, remapping keys...
Load message: <All keys matched successfully>
Model loaded successfully!


Zelle 5 – Alle Funktionen (Blur, Padding, bilineare Extraktion)

In [5]:
def apply_motion_blur(frames, center_idx, window_size, gamma=2.2):
    half = window_size // 2
    start, end = max(0, center_idx - half), min(len(frames), center_idx + half + 1)
    window = frames[start:end].astype(np.float32) / 255.0
    return ((window ** gamma).mean(axis=0) ** (1.0 / gamma) * 255).astype(np.uint8)

def blur_video(frames, window_size):
    return np.stack([apply_motion_blur(frames, i, window_size) for i in range(len(frames))])

BLUR_LEVELS = {"none": 1, "medium": 3, "strong": 7}

def pad_to_112(v):
    H, W = v.shape[-2:]
    return F.pad(v, (0, (112 - W % 112) % 112, 0, (112 - H % 112) % 112))

def sample_dense(field, y, x):
    f = field.float()
    if f.dim() == 3:
        f = f.unsqueeze(-1)
    H, W = f.shape[1], f.shape[2]
    x = float(np.clip(x, 0, W - 1)); y = float(np.clip(y, 0, H - 1))
    x0, y0 = int(np.floor(x)), int(np.floor(y))
    x1, y1 = min(x0 + 1, W - 1), min(y0 + 1, H - 1)
    wx, wy = x - x0, y - y0
    v = (f[:, y0, x0] * (1 - wx) * (1 - wy) + f[:, y0, x1] * wx * (1 - wy)
         + f[:, y1, x0] * (1 - wx) * wy + f[:, y1, x1] * wx * wy)
    return v.squeeze(-1)

def track_query_points_cowtracker(model, video_chw, query_points):
    T, N = video_chw.shape[0], len(query_points)
    pred_tracks = np.zeros((N, T, 2), dtype=np.float32)
    pred_vis = np.zeros((N, T), dtype=bool)
    for qt in np.unique(query_points[:, 0]).astype(int):
        idx = np.where(query_points[:, 0].astype(int) == qt)[0]
        with torch.no_grad():
            pf = model(pad_to_112(video_chw[qt:]).half())
        for gi in idx:
            _, y, x = query_points[gi]
            pred_tracks[gi, qt:] = sample_dense(pf['track'][0], y, x).cpu().numpy()
            pred_vis[gi, qt:] = sample_dense(pf['vis'][0], y, x).cpu().numpy() > 0.5
        if qt > 0:
            with torch.no_grad():
                pb = model(pad_to_112(video_chw[:qt + 1].flip(0)).half())
            for gi in idx:
                _, y, x = query_points[gi]
                pred_tracks[gi, :qt + 1] = sample_dense(pb['track'][0], y, x).cpu().numpy()[::-1]
                pred_vis[gi, :qt + 1] = sample_dense(pb['vis'][0], y, x).cpu().numpy()[::-1] > 0.5
    return pred_tracks, pred_vis

def to_uint8(video_batch):
    return ((video_batch[0] + 1) / 2 * 255).astype(np.uint8)

Zelle 6 – Baseline-Schnelltest, 5 Videos unverblurrt (Ziel: Richtung 65,5 AJ / 78,0 δ)

In [6]:
res = []
for i, sample in enumerate(create_davis_dataset('tapvid_davis/tapvid_davis.pkl', query_mode='first')):
    if i >= 5: break
    b = sample['davis']; qp = b['query_points'][0]
    chw = torch.from_numpy(to_uint8(b['video'])).permute(0, 3, 1, 2).float().cuda()
    tr, vi = track_query_points_cowtracker(model, chw, qp)
    s = compute_tapvid_metrics(b['query_points'], b['occluded'], b['target_points'], (~vi)[None], tr[None], query_mode='first')
    res.append({k: float(np.sum(v)) * 100 for k, v in s.items()})
    print(f"video {i}: AJ {res[-1]['average_jaccard']:.1f} | δ {res[-1]['average_pts_within_thresh']:.1f} | OA {res[-1]['occlusion_accuracy']:.1f}")
import pandas as pd
print(pd.DataFrame(res)[["average_jaccard", "average_pts_within_thresh", "occlusion_accuracy"]].mean().round(1))

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


video 0: AJ 60.4 | δ 67.9 | OA 100.0
video 1: AJ 36.8 | δ 60.7 | OA 81.1
video 2: AJ 55.3 | δ 69.8 | OA 94.9
video 3: AJ 51.7 | δ 68.9 | OA 88.2
video 4: AJ 67.9 | δ 87.8 | OA 91.5
average_jaccard              54.4
average_pts_within_thresh    71.0
occlusion_accuracy           91.1
dtype: float64


Hochskalierung der Videos:

In [7]:
def track_query_points_cowtracker_resized(model, video_chw, query_points, target=336):
    T, N = video_chw.shape[0], len(query_points)
    H, W = video_chw.shape[-2:]
    sy, sx = target / H, target / W
    vid = F.interpolate(video_chw, size=(target, target), mode="bilinear", align_corners=False)
    pred_tracks = np.zeros((N, T, 2), dtype=np.float32)
    pred_vis = np.zeros((N, T), dtype=bool)
    for qt in np.unique(query_points[:, 0]).astype(int):
        idx = np.where(query_points[:, 0].astype(int) == qt)[0]
        with torch.no_grad():
            pf = model(vid[qt:].half())
        for gi in idx:
            _, y, x = query_points[gi]
            tr = sample_dense(pf['track'][0], y * sy, x * sx).cpu().numpy()
            pred_tracks[gi, qt:] = tr / np.array([sx, sy], dtype=np.float32)  # (x,y) zurück in 256er-Raum
            pred_vis[gi, qt:] = sample_dense(pf['vis'][0], y * sy, x * sx).cpu().numpy() > 0.5
        if qt > 0:
            with torch.no_grad():
                pb = model(vid[:qt + 1].flip(0).half())
            for gi in idx:
                _, y, x = query_points[gi]
                tr = sample_dense(pb['track'][0], y * sy, x * sx).cpu().numpy()[::-1]
                pred_tracks[gi, :qt + 1] = tr / np.array([sx, sy], dtype=np.float32)
                pred_vis[gi, :qt + 1] = sample_dense(pb['vis'][0], y * sy, x * sx).cpu().numpy()[::-1] > 0.5
    return pred_tracks, pred_vis

for target in [336, 448]:
    res = []
    for i, sample in enumerate(create_davis_dataset('tapvid_davis/tapvid_davis.pkl', query_mode='first')):
        if i >= 5: break
        b = sample['davis']; qp = b['query_points'][0]
        chw = torch.from_numpy(to_uint8(b['video'])).permute(0, 3, 1, 2).float().cuda()
        tr, vi = track_query_points_cowtracker_resized(model, chw, qp, target=target)
        s = compute_tapvid_metrics(b['query_points'], b['occluded'], b['target_points'], (~vi)[None], tr[None], query_mode='first')
        res.append({k: float(np.sum(v)) * 100 for k, v in s.items()})
    m = pd.DataFrame(res)[["average_jaccard", "average_pts_within_thresh", "occlusion_accuracy"]].mean()
    print(f"target {target}: AJ {m['average_jaccard']:.1f} | δ {m['average_pts_within_thresh']:.1f} | OA {m['occlusion_accuracy']:.1f}")

target 336: AJ 59.1 | δ 75.8 | OA 90.9
target 448: AJ 61.8 | δ 78.3 | OA 91.5


Lauf für Experiment 2

In [8]:
import pandas as pd, numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
base = "/content/drive/MyDrive/ba_results/"

def longest_run(row):
    best = cur = 0
    for v in row:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best

def compute_exp2_metrics(pred_tracks, pred_visibility, target_points, occluded, query_times):
    """Identical to the CoTracker3 / TAPNext v4 function, plus the length ratio of the earlier evaluation."""
    occ = occluded[0] if occluded.ndim == 3 else occluded
    N, T = occ.shape
    eval_mask = np.arange(T)[None, :] >= query_times[:, None].astype(int)
    gt_vis = ~occ & eval_mask
    gt_occ = occ & eval_mask
    both_vis = gt_vis[:, 1:] & gt_vis[:, :-1]
    pred_jump = np.linalg.norm(pred_tracks[:, 1:] - pred_tracks[:, :-1], axis=-1)
    gt_jump = np.linalg.norm(target_points[:, 1:] - target_points[:, :-1], axis=-1)
    pred_max = np.array([pj[m].max() if m.any() else np.nan for pj, m in zip(pred_jump, both_vis)])
    gt_max = np.array([gj[m].max() if m.any() else np.nan for gj, m in zip(gt_jump, both_vis)])
    valid = ~np.isnan(pred_max)
    pv = pred_visibility.astype(bool)
    pred_len = np.array([longest_run(r) for r in pv])
    gt_len = np.array([longest_run(r) for r in ~occ])
    return {
        "outlier_magnitude": float(np.nanmean(pred_max)),
        "outlier_ratio_vs_gt": float(np.nanmean(pred_max[valid] / np.maximum(gt_max[valid], 1e-3))),
        "false_visible_rate": float((pv & gt_occ).sum() / max(gt_occ.sum(), 1)),
        "false_occluded_rate": float((~pv & gt_vis).sum() / max(gt_vis.sum(), 1)),
        "trajectory_length_ratio": float((pred_len / np.maximum(gt_len, 1)).mean()),
    }

results_path = base + "cowtracker_all_videos_v4.csv"
all_results = []
for video_idx, sample in enumerate(create_davis_dataset('tapvid_davis/tapvid_davis.pkl', query_mode='first')):
    b = sample['davis']; qp = b['query_points'][0]; frames = to_uint8(b['video'])
    for level_name, window in BLUR_LEVELS.items():
        chw = torch.from_numpy(blur_video(frames, window)).permute(0, 3, 1, 2).float().cuda()
        tr, vi = track_query_points_cowtracker_resized(model, chw, qp, target=448)
        s = compute_tapvid_metrics(b['query_points'], b['occluded'], b['target_points'],
                                   (~vi)[None], tr[None], query_mode='first')
        row = {"video": video_idx, "model": "CoWTracker", "blur": level_name}
        row.update({k: float(np.sum(v)) * 100 for k, v in s.items()})
        row.update(compute_exp2_metrics(tr, vi, b['target_points'][0], b['occluded'], qp[:, 0]))
        all_results.append(row)
        if video_idx == 22 and level_name == "strong":
            np.save(base + "cowtracker_traj_video22_strong.npy", tr)
    pd.DataFrame(all_results).to_csv(results_path, index=False)
    print(f"Video {video_idx}/29 done")

df = pd.DataFrame(all_results)
print(df.groupby("blur")[["average_jaccard", "occlusion_accuracy", "average_pts_within_thresh",
                          "outlier_magnitude", "outlier_ratio_vs_gt", "false_visible_rate",
                          "false_occluded_rate", "trajectory_length_ratio"]].mean().round(3))

Mounted at /content/drive
Video 0/29 done
Video 1/29 done
Video 2/29 done
Video 3/29 done
Video 4/29 done
Video 5/29 done
Video 6/29 done
Video 7/29 done
Video 8/29 done
Video 9/29 done
Video 10/29 done
Video 11/29 done
Video 12/29 done
Video 13/29 done
Video 14/29 done
Video 15/29 done
Video 16/29 done
Video 17/29 done
Video 18/29 done
Video 19/29 done
Video 20/29 done
Video 21/29 done
Video 22/29 done
Video 23/29 done
Video 24/29 done
Video 25/29 done
Video 26/29 done
Video 27/29 done
Video 28/29 done
Video 29/29 done
        average_jaccard  occlusion_accuracy  average_pts_within_thresh  \
blur                                                                     
medium           52.114              89.899                     65.819   
none             65.325              92.113                     78.781   
strong           35.252              85.606                     48.184   

        outlier_magnitude  outlier_ratio_vs_gt  false_visible_rate  \
blur                             

Zelle 7 – Vollauf (erst starten, wenn Zelle 6 die Lücke verkleinert hat)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/ba_results', exist_ok=True)
results_path = "/content/drive/MyDrive/ba_results/cowtracker_all_videos_v3.csv"

all_results = []
for video_idx, sample in enumerate(create_davis_dataset('tapvid_davis/tapvid_davis.pkl', query_mode='first')):
    b = sample['davis']; qp = b['query_points'][0]; frames = to_uint8(b['video'])
    for level_name, window in BLUR_LEVELS.items():
        chw = torch.from_numpy(blur_video(frames, window)).permute(0, 3, 1, 2).float().cuda()
        tr, vi = track_query_points_cowtracker_resized(model, chw, qp, target=448)
        s = compute_tapvid_metrics(b['query_points'], b['occluded'], b['target_points'], (~vi)[None], tr[None], query_mode='first')
        row = {"video": video_idx, "model": "CoWTracker", "blur": level_name}
        row.update({k: float(np.sum(v)) * 100 for k, v in s.items()})
        all_results.append(row)
    pd.DataFrame(all_results).to_csv(results_path, index=False)
    print(f"Video {video_idx}/29 fertig")

df = pd.DataFrame(all_results)
print(df.groupby("blur")[["average_jaccard", "occlusion_accuracy", "average_pts_within_thresh"]].mean().round(1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Video 0/29 fertig
Video 1/29 fertig
Video 2/29 fertig
Video 3/29 fertig
Video 4/29 fertig
Video 5/29 fertig
Video 6/29 fertig
Video 7/29 fertig
Video 8/29 fertig
Video 9/29 fertig
Video 10/29 fertig
Video 11/29 fertig
Video 12/29 fertig
Video 13/29 fertig
Video 14/29 fertig
Video 15/29 fertig
Video 16/29 fertig
Video 17/29 fertig
Video 18/29 fertig
Video 19/29 fertig
Video 20/29 fertig
Video 21/29 fertig
Video 22/29 fertig
Video 23/29 fertig
Video 24/29 fertig
Video 25/29 fertig
Video 26/29 fertig
Video 27/29 fertig
Video 28/29 fertig
Video 29/29 fertig
        average_jaccard  occlusion_accuracy  average_pts_within_thresh
blur                                                                  
medium             52.1                89.9                       65.8
none               65.3                92.1                       78.8
strong             35.3    